In [0]:
# ============================================================
# STEG 1 – Importera bibliotek och hämta data
# ============================================================

# Importerar MLflow (används senare för att logga modeller och resultat osv...)
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Importerar pandas, hanterar/skapar tabeller
import pandas as pd

# Importerar numpy för numeriska beräkningar :)
import numpy as np

# Importerar funktionen som delar upp datan i träning och test
from sklearn.model_selection import train_test_split

# Importerar Random Forest (supervised modell) Innebär att den kommer tränas med labels 
from sklearn.ensemble import RandomForestClassifier

# Importerar Isolation Forest (unsupervised modell) Innebär att den kommer tränas utan labels
from sklearn.ensemble import IsolationForest

# Importerar LabelEncoder för att omvandla text till siffror
from sklearn.preprocessing import LabelEncoder

# Importerar utvärderingsmått
from sklearn.metrics import classification_report, roc_auc_score

# Skriver ut MLflow-version (endast för att se om jag har en gammal version) // felsökning // 1 timme slösades här :)
print("MLflow version:", mlflow.__version__)


# ----- Hämta data från Snowflake -----

# Läser in din feature-tabell från Snowflake via Databricks 
# OBS namnet namnet är inte samma som i Snowflake, det var kul att upptäcka!
df = spark.table("snowflakev1_catalog.dbt_aleanderdp.FEATURE_MODEL").toPandas()

# Visar hur många rader och kolumner datan har
print("Shape:", df.shape)

# Visar alla kolumnnamn
print("\nKolumner:")
print(df.columns.tolist())

# Visar hur stor andel som är malicious (1) vs benign (0) True/False
print("\nKlassfördelning:")
print(df['IS_MALICIOUS'].value_counts(normalize=True).round(4))

# Visar de första 5 raderna, endast för kontroll av data och hur den ser ut
df.head()

MLflow version: 3.15.1
Shape: (3209, 18)

Kolumner:
['UID', 'TS', 'DURATION', 'ORIG_BYTES', 'RESP_BYTES', 'ORIG_PKTS', 'RESP_PKTS', 'ORIG_IP_BYTES', 'RESP_IP_BYTES', 'BYTES_RATIO', 'PKTS_RATIO', 'IS_LOCAL_ORIG', 'IS_LOCAL_RESP', 'PROTOCOL', 'SERVICE', 'CONN_STATE', 'IS_MALICIOUS', 'DETAILED_LABEL']

Klassfördelning:
IS_MALICIOUS
0    0.995
1    0.005
Name: proportion, dtype: float64


,UID,TS,DURATION,ORIG_BYTES,RESP_BYTES,ORIG_PKTS,RESP_PKTS,ORIG_IP_BYTES,RESP_IP_BYTES,BYTES_RATIO,PKTS_RATIO,IS_LOCAL_ORIG,IS_LOCAL_RESP,PROTOCOL,SERVICE,CONN_STATE,IS_MALICIOUS,DETAILED_LABEL
0,CSQG794riQ4XnzTxP2,1538478769.600293,5,78,0,2,0,134,0,None,None,0,0,udp,dns,S0,0,-
1,COTbdG2BhtGBlmf6r,1538478779.610847,0,90,90,2,2,146,146,1.000000,1.000000,0,0,udp,dns,SF,0,-
2,CP48WJ2HOnLuGtr5kb,1538478789.630642,0,90,90,2,2,146,146,1.000000,1.000000,0,0,udp,dns,SF,0,-
3,CeTMJi2TydRSaVdsG4,1538478779.620088,5,78,0,2,0,134,0,None,None,0,0,udp,dns,S0,0,-
4,CZ6ne24AN9WAg9XA9d,1538478799.645444,0,90,90,2,2,146,146,1.000000,1.000000,0,0,udp,dns,SF,0,-


In [0]:
# ============================================================
# STEG 2 – Förbered features för modellerna
# ============================================================

# Lista över de numeriska features vi ska använda
# (använder VERSALER eftersom kolumnnamnen i Snowflake är stora)
numeric_features = [
    'DURATION',
    'ORIG_BYTES',
    'RESP_BYTES',
    'ORIG_PKTS',
    'RESP_PKTS',
    'ORIG_IP_BYTES',
    'RESP_IP_BYTES',
    'BYTES_RATIO',
    'PKTS_RATIO',
    'IS_LOCAL_ORIG',
    'IS_LOCAL_RESP'
]

# Lista över de kategoriska (text) features
categorical_features = [
    'PROTOCOL',
    'SERVICE',
    'CONN_STATE'
]

# Fyller saknade numeriska värden med 0 (modellerna klarar endast nummer, inte NaN)
df[numeric_features] = df[numeric_features].fillna(0)

# Fyller saknade kategoriska värden med texten 'unknown'
df[categorical_features] = df[categorical_features].fillna('unknown')

# Skapar en dictionary där vi sparar LabelEncoders
# (behövs om vi senare vill transformera ny data på samma sätt)
le_dict = {}

# Loopar igenom varje kategorisk kolumn och omvandlar text till siffror
for col in categorical_features:
    le = LabelEncoder()                              # skapar en ny encoder
    df[col] = le.fit_transform(df[col].astype(str))  # tränar och omvandlar kolumnen
    le_dict[col] = le                                # sparar encodern

# Skapar feature-matrisen X (alla features vi ska träna på)
X = df[numeric_features + categorical_features]

# Skapar target-vektorn y (det vi vill förutsäga: 0 = benign, 1 = malicious)
y = df['IS_MALICIOUS'].astype(int)

# Delar upp datan i tränings- och testmängd (75 % träning, 25 % test)
# stratify=y ser till att andelen malicious blir ungefär densamma i båda mängderna
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,      # 25 % av datan blir testmängd
    random_state=42,     # gör att uppdelningen blir densamma varje gång
    stratify=y           # behåller klassfördelningen
)

# Skriver ut storleken på tränings- och testmängden
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# Skriver ut andelen malicious i hela datasetet
print(f"Malicious ratio i hela datan: {y.mean():.4f}")

# Visar de första raderna i X så vi ser att allt ser bra ut
X.head()

Train: (2406, 14) | Test: (803, 14)
Malicious ratio i hela datan: 0.0050


,DURATION,ORIG_BYTES,RESP_BYTES,ORIG_PKTS,RESP_PKTS,ORIG_IP_BYTES,RESP_IP_BYTES,BYTES_RATIO,PKTS_RATIO,IS_LOCAL_ORIG,IS_LOCAL_RESP,PROTOCOL,SERVICE,CONN_STATE
0,5,78,0,2,0,134,0,0,0,0,0,1,1,2
1,0,90,90,2,2,146,146,1.000000,1.000000,0,0,1,1,4
2,0,90,90,2,2,146,146,1.000000,1.000000,0,0,1,1,4
3,5,78,0,2,0,134,0,0,0,0,0,1,1,2
4,0,90,90,2,2,146,146,1.000000,1.000000,0,0,1,1,4


In [0]:
# ============================================================
# STEG 3 – Random Forest, Supervised
# ============================================================

# Startar ett nytt MLflow-experiment/run med ett beskrivande namn
with mlflow.start_run(run_name="RandomForest_CTU_IoT_v2") as run:

    # Skapar en Random Forest-klassificerare med valda hyperparametrar
    # (justerade för att undvika 100% accuracy)
    rf = RandomForestClassifier(
        n_estimators=60,               # antal träd i "skogen"
        max_depth=10,                  # max djup på varje träd (vill inte överanpassa)
        min_samples_leaf=5,            # minsta antal observationer i ett löv
        max_features='sqrt',           # använder bara en del av features per träd
        class_weight=None,             # ingen extra viktning (viktigt för mer realistiska resultat)
        random_state=42,               # Svaret på universum är 42
        n_jobs=-1                      # använder alla tillgängliga CPU-kärnor
    )

    # Tränar modellen på träningsdatan (Fit = Träning)
    # Här lär sig modellen mönster som skiljer benign från malicious
    rf.fit(X_train, y_train)

    # Gör prediktioner (predict) på testdatan → ger 0 eller 1
    y_pred = rf.predict(X_test)

    # Hämtar sannolikheter för klassen 1 (malicious)
    y_proba = rf.predict_proba(X_test)[:, 1]

    # Prediktioner (sannolikheter) — tillakt för att kunna skapa ROC-kurva
    y_pred_proba = rf.predict_proba(X_test)[:, 1]

    # Skapar en detaljerad utvärderingsrapport (precision, recall, f1 per klass)
    report = classification_report(y_test, y_pred, output_dict=True)

    # Beräknar ROC-AUC (bra mått när klasserna är obalanserade)
    auc = roc_auc_score(y_test, y_proba)

    # Skriver ut resultaten här på plats
    print("=== Random Forest (v2) ===")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {auc:.4f}")

    # Loggar hyperparametrar till MLflow
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 60)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("min_samples_leaf", 5)
    mlflow.log_param("class_weight", "None")

    # Loggar utvärderingsmått till MLflow
    mlflow.log_metric("roc_auc", auc)
    mlflow.log_metric("precision_malicious", report['1']['precision'])
    mlflow.log_metric("recall_malicious", report['1']['recall'])
    mlflow.log_metric("f1_malicious", report['1']['f1-score'])
    mlflow.log_metric("accuracy", report['accuracy'])

    # Skapar en signatur som beskriver modellens input/output
    signature = infer_signature(X_train, rf.predict(X_train))

    # Loggar själva modellen till MLflow (kan senare laddas ner eller deployas)
    mlflow.sklearn.log_model(rf, "model", signature=signature)

    # Beräknar feature importance (vilka features modellen tycker är viktigast)
    importance = pd.Series(rf.feature_importances_, index=X.columns)\
                    .sort_values(ascending=False)

    # Skriver ut de 10 viktigaste features
    print("\nTop 10 viktigaste features:")
    print(importance.head(10))

=== Random Forest (v2) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       799
           1       1.00      0.25      0.40         4

    accuracy                           1.00       803
   macro avg       1.00      0.62      0.70       803
weighted avg       1.00      1.00      1.00       803

ROC-AUC: 1.0000


2026/08/09 18:28:35 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
2026/08/09 18:28:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-5cfebbcb-2971.cloud.databricks.com/ml/experiments/151263118468432/models/m-4141aa4931be445693ea30f2808bcbea?o=7474648211871361



Top 10 viktigaste features:
PROTOCOL         0.432066
ORIG_IP_BYTES    0.363260
ORIG_BYTES       0.065303
ORIG_PKTS        0.047571
PKTS_RATIO       0.025108
DURATION         0.023404
CONN_STATE       0.017923
RESP_IP_BYTES    0.009057
BYTES_RATIO      0.008708
RESP_BYTES       0.004237
dtype: float64


In [0]:
# ============================================================
# STEG 4 – Träna Isolation Forest
# ============================================================

# Startar ett nytt MLflow-run för Isolation Forest
with mlflow.start_run(run_name="IsolationForest_CTU_IoT") as run:

    # Tar endast ut de rader som är benign (y_train == 0)
    # Isolation Forest tränas bäst på "normal" trafik
    X_benign = X_train[y_train == 0]

    # Skapar Isolation Forest-modellen
    iso = IsolationForest(
        n_estimators=500,        # antal träd
        contamination=0.04,      # ungefärlig andel anomalier vi förväntar oss
        random_state=42,         # Är svaret på universum fortfarande 42?
        n_jobs=-1                # använd alla CPU-kärnor
    )

    # Tränar modellen enbart på benign trafik (Fit = Träning)
    # Modellen lär sig hur "normal" trafik ser ut
    iso.fit(X_benign)

    # Gör prediktioner på testdatan
    # Isolation Forest returnerar -1 för anomaly och 1 för normal
    # Vi omvandlar till 1 = malicious, 0 = benign
    y_pred_iso = np.where(iso.predict(X_test) == -1, 1, 0)

    # Skapar utvärderingsrapport
    report_iso = classification_report(y_test, y_pred_iso, output_dict=True)

    # Skriver ut resultaten
    print("=== Isolation Forest ===")
    print(classification_report(y_test, y_pred_iso))

    # Loggar parametrar och metrics till MLflow
    mlflow.log_param("model_type", "IsolationForest")
    mlflow.log_param("contamination", 0.04)
    mlflow.log_param("n_estimators", 500)
    mlflow.log_metric("precision_malicious", report_iso['1']['precision'])
    mlflow.log_metric("recall_malicious", report_iso['1']['recall'])
    mlflow.log_metric("f1_malicious", report_iso['1']['f1-score'])
    mlflow.log_metric("accuracy", report_iso['accuracy'])

    # Loggar Isolation Forest-modellen till MLflow
    mlflow.sklearn.log_model(iso, "model")

=== Isolation Forest ===
              precision    recall  f1-score   support

           0       1.00      0.96      0.98       799
           1       0.03      0.25      0.06         4

    accuracy                           0.96       803
   macro avg       0.51      0.61      0.52       803
weighted avg       0.99      0.96      0.98       803



2026/08/09 18:28:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-5cfebbcb-2971.cloud.databricks.com/ml/experiments/151263118468432/models/m-1d04dc2453644f8fb024836e9f90631d?o=7474648211871361
2026/08/09 18:28:56 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.15.1/ml/model/signatures.html for instructions on setting signature on models.


In [0]:
# ============================================================
# STEG 5 – Spara modeller + skapa nedladdningsknappar
# ============================================================

import joblib
import os
import base64
from IPython.display import HTML, display

# Spara modellerna i /tmp
os.makedirs("/tmp/models", exist_ok=True)

joblib.dump(rf, "/tmp/models/random_forest_model.pkl")
joblib.dump(iso, "/tmp/models/isolation_forest_model.pkl")
joblib.dump(le_dict, "/tmp/models/label_encoders.pkl")

print("Modellerna är sparade.\n")

def create_download_link(file_path, file_name):
    """Skapar en nedladdningslänk/knapp för en fil"""
    with open(file_path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    
    # Gör en HTML-kod som innehåller en nedladdningsknapp för lättare nerladdning
    html = f'''
    <a download="{file_name}" 
       href="data:application/octet-stream;base64,{b64}" 
       style="
           display:inline-block;
           padding:10px 18px;
           margin:6px 0;
           background-color:#0d6efd;
           color:white;
           text-decoration:none;
           border-radius:6px;
           font-weight:bold;
           font-family:sans-serif;
       ">
       ⬇ Ladda ner {file_name}
    </a>
    '''
    return HTML(html)

# Visa knapparna (label_encoders är inte modell, kan användas av andra användare om de vill träna modellerna ytterligare)
display(create_download_link("/tmp/models/random_forest_model.pkl", "random_forest_model.pkl"))
display(create_download_link("/tmp/models/isolation_forest_model.pkl", "isolation_forest_model.pkl"))
display(create_download_link("/tmp/models/label_encoders.pkl", "label_encoders.pkl"))

Modellerna är sparade.



In [0]:
# ============================================================
# STEG 6 – Skapa Slack-rapport + komplett Python-fil
# ============================================================

# Pga att min dashboard/workspace/notebook inte har åtkomst till internet och inte kan använda sig av webhooks eller API-calls, har jag valt att skapa en lokal lösning. Denna lösning består av att denna cell skapar ett Python script som skickar rapporten till Slack med en Webhook och använder sig av en GROK API sammanställa all info om medellerna och deras prestanda i text format. Därefter görs ett till API-call som summerar all info från GROK tidigare sammanställning av modellerna. Båda dessa AI-summeringarna skickas vidare till Slack.

import json
import base64
from IPython.display import HTML, display

print("Skapar Slack-rapport och komplett Python-fil...")

# Bygg Slack-rapporten från modeller

top_features = importance.head(10)
feature_text = "\n".join(
    f"• {feature}: {value:.4f}"
    for feature, value in top_features.items()
)

# Allt innehåll till slack rapporten
slack_text = (
    "*CTU IoT Malware – Modellrapport*\n\n"
    "*Random Forest – Supervised*\n"
    f"• Accuracy: {report['accuracy']:.2%}\n"
    f"• Precision (malicious): {report['1']['precision']:.2%}\n"
    f"• Recall (malicious): {report['1']['recall']:.2%}\n"
    f"• F1-score (malicious): {report['1']['f1-score']:.2%}\n"
    f"• ROC-AUC: {auc:.4f}\n\n"
    "*Isolation Forest – Unsupervised*\n"
    f"• Accuracy: {report_iso['accuracy']:.2%}\n"
    f"• Precision (malicious): {report_iso['1']['precision']:.2%}\n"
    f"• Recall (malicious): {report_iso['1']['recall']:.2%}\n"
    f"• F1-score (malicious): {report_iso['1']['f1-score']:.2%}\n\n"
    "*Top 10 viktigaste features – Random Forest*\n"
    f"{feature_text}\n\n"
    "*MLflow*\n"
    "Random Forest och Isolation Forest har tränats "
    "och loggats till MLflow.\n\n"
    "Rapport genererad automatiskt."
)

report_obj = {"slack_text": slack_text}

# Spara Slack-rapporten som JSON

json_path = "/tmp/slack_report.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report_obj, f, ensure_ascii=False, indent=2)

# Print för syns skull
print("✔ Slack-rapport sparad i /tmp/slack_report.json")


# 3. Skapa komplett Python-fil som körs lokalt

python_script = r'''
import json
import requests
import socket
from getpass import getpass

# ============================================================
# DNS-check (lokalt)
# ============================================================

def check_dns(host="hooks.slack.com"):
    try:
        socket.gethostbyname(host)
        return True
    except Exception as e:
        print(f" DNS-fel: {e}")
        return False

# ============================================================
# Läs ML-rapporten
# ============================================================

with open("slack_report.json", "r", encoding="utf-8") as f:
    data = json.load(f)

slack_text = data["slack_text"]

# ============================================================
# Hämta API-nycklar ((GETPASS))
# ============================================================

webhook_url = getpass("Klistra in din Slack Webhook URL: ")
grok_api_key = getpass("Klistra in din GROK API-nyckel: ")

print("\n Kör DNS-test...")
if not check_dns():
    print(" DNS fungerar inte. Avslutar.")
    exit(1)

print("✔ DNS OK\n")

# ============================================================
# Skicka rapporten till Grok API (första sammanfattningen)
# ============================================================

print("🤖 Skickar rapporten till Grok AI...")

grok_url = "https://api.x.ai/v1/chat/completions"

prompt = f"""
Sammanfatta följande ML-resultat och jämför Random Forest och Isolation Forest.
Beskriv styrkor, svagheter och vilken modell som är mest lämplig för IoT-malware-detektering.

Rapport:
{slack_text}
"""

payload = {
    "model": "grok-4.5",
    "messages": [
        {"role": "user", "content": prompt}
    ]
}

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {grok_api_key}"
}

grok_response = requests.post(grok_url, json=payload, headers=headers)

if grok_response.status_code != 200:
    print(" Grok API-fel:", grok_response.text)
    exit(1)

summary = grok_response.json()["choices"][0]["message"]["content"]

print("\n Grok sammanfattning:\n")
print(summary)

# ============================================================
# Grok sammanfattar sin egen sammanfattning (meta-summary)
# ============================================================

print("\n Skickar sammanfattningen till Grok igen för meta-summary...")

meta_prompt = f"""
Sammanfatta följande text extremt kortfattat.
Fokusera på:
- huvudpoängen
- modelljämförelsen
- slutsatsen
- rekommendationen

Text:
{summary}
"""

meta_payload = {
    "model": "grok-4.5",
    "messages": [
        {"role": "user", "content": meta_prompt}
    ]
}

meta_response = requests.post(grok_url, json=meta_payload, headers=headers)

if meta_response.status_code != 200:
    print(" Grok API-fel (meta-summary):", meta_response.text)
    exit(1)

meta_summary = meta_response.json()["choices"][0]["message"]["content"]

print("\n Grok meta-summary:\n")
print(meta_summary)


# Skicka originalrapporten till Slack


print("\n📤 Skickar originalrapporten till Slack...")

original_message = {
    "text": f"*CTU IoT Malware – Modellrapport*\n\n{slack_text}"
}

response = requests.post(webhook_url, json=original_message)

if response.status_code == 200:
    print("✔ Originalrapport skickad!")
else:
    print("❌ Slack-fel:", response.text)


# Skicka AI-sammanfattningen till Slack


print("\n📤 Skickar AI-sammanfattningen till Slack...")

summary_message = {
    "text": f"*AI-sammanfattning av ML-resultat*\n\n{summary}"
}

response = requests.post(webhook_url, json=summary_message)

if response.status_code == 200:
    print("✔ AI-sammanfattning skickad!")
else:
    print("❌ Slack-fel:", response.text)


# Skicka meta-summary till Slack (sammanfattning nummer två)


print("\n Skickar meta-summary till Slack...")

meta_message = {
    "text": f"*AI Meta-Summary (Grok på Grok)*\n\n{meta_summary}"
}

response = requests.post(webhook_url, json=meta_message)

if response.status_code == 200:
    print(" Meta-summary skickad!")
else:
    print(" Slack-fel:", response.text)

print("\n Klart! Alla tre rapporterna är skickade till Slack.")
'''

py_path = "/tmp/send_full_report.py"
with open(py_path, "w", encoding="utf-8") as f:
    f.write(python_script)

print("✔ Python-fil sparad i /tmp/send_full_report.py")



# 4. Skapa nedladdningsknappar


def create_download_button(path, filename):
    with open(path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()

    html = f'''
    <a download="{filename}"
       href="data:application/octet-stream;base64,{b64}"
       style="
           display:inline-block;
           padding:10px 18px;
           margin:6px 0;
           background-color:#0d6efd;
           color:white;
           text-decoration:none;
           border-radius:6px;
           font-weight:bold;
           font-family:sans-serif;
       ">
       ⬇ Ladda ner {filename}
    </a>
    '''
    return HTML(html)

display(create_download_button(json_path, "slack_report.json"))
display(create_download_button(py_path, "send_full_report.py"))

print(" Klart! Ladda ner filerna och kör send_full_report.py lokalt.")


Skapar Slack-rapport och komplett Python-fil...
✔ Slack-rapport sparad i /tmp/slack_report.json
✔ Python-fil sparad i /tmp/send_full_report.py


 Klart! Ladda ner filerna och kör send_full_report.py lokalt.


In [0]:
# ============================================================
# DATBRICKS DASHBOARD-CELL (EN CELL)
# - Visar jämförelse mellan Random Forest (RF) och Isolation Forest (ISO)
# ============================================================

# ---------- Importer ----------
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings("ignore")

sns.set(style="whitegrid")

# ---------- Hjälpfunktioner (säkra kontroller) ----------
def is_nonempty_array_like(x):
    """
    Returnerar True om x är en array/serie/lista med minst ett element.
    Returnerar False om x är None eller tom.
    Använd för att undvika ValueError vid if x: på array/serie.
    """
    if x is None:
        return False
    if isinstance(x, (pd.Series, pd.DataFrame)):
        return not x.empty
    if isinstance(x, np.ndarray):
        return x.size > 0
    if isinstance(x, (list, tuple)):
        return len(x) > 0
    return True

def first_nonempty(*keys):
    """
    Leta i globals() efter första icke-tomma värdet bland angivna nycklar.
    Returnerar värdet eller None.
    """
    for k in keys:
        v = globals().get(k, None)
        if is_nonempty_array_like(v):
            return v
    return None

def safe_auc(y_true, scores):
    """Beräkna ROC AUC och returnera (auc, fpr, tpr) eller (None, None, None) vid fel."""
    try:
        fpr, tpr, _ = roc_curve(y_true, scores)
        return auc(fpr, tpr), fpr, tpr
    except Exception:
        return None, None, None

def safe_pr(y_true, scores):
    """Beräkna PR AUC och returnera (pr_auc, precision, recall) eller (None, None, None)."""
    try:
        precision, recall, _ = precision_recall_curve(y_true, scores)
        return average_precision_score(y_true, scores), precision, recall
    except Exception:
        return None, None, None

# ---------- Widgets (kör en gång) ----------
try:
    dbutils.widgets.removeAll()
except Exception:
    pass

# Modellval (behåll för flexibilitet), tröskel, TopN och permutation toggle
dbutils.widgets.dropdown("ModelSelect", "Both", ["Both","RandomForest","IsolationForest"], "Visa modell")
dbutils.widgets.text("Threshold", "0.5", "Tröskel (0-1) för binär klassning (används för RF och ISO score->binär om önskas)")
dbutils.widgets.dropdown("TopN", "10", [str(i) for i in [5,10,15,20]], "Top N features")
dbutils.widgets.dropdown("ShowPermutation", "no", ["yes","no"], "Visa permutation importance (kan vara tungt)")

# ---------- Läs widgetvärden ----------
model_choice = dbutils.widgets.get("ModelSelect")
threshold = float(dbutils.widgets.get("Threshold"))
top_n = int(dbutils.widgets.get("TopN"))
show_perm = dbutils.widgets.get("ShowPermutation") == "yes"

print(f"Inställningar: model={model_choice}, threshold={threshold}, TopN={top_n}, Perm={show_perm}")

# ---------- Hämta variabler säkert från globals ----------
# RF: försök hämta sannolikheter och prediktioner från vanliga namn
rf_scores = first_nonempty("y_proba", "y_pred_proba")   # sannolikheter för klass 1 (RF)
rf_preds = first_nonempty("y_pred",)                    # binära prediktioner (RF)
rf_importance = globals().get("importance", None)       # RF feature importance (pd.Series)

# ISO: hämta om finns (kan vara None)
iso_scores = first_nonempty("iso_scores_norm", "iso_scores")  # normaliserade ISO-scores (0-1) om tillgängliga
iso_preds = first_nonempty("y_pred_iso",)                     # ISO binära prediktioner (0/1) om tillgängliga

# Debug-utskrifter för typ/shape — användbart vid felsökning
print("DEBUG: rf_scores type/shape:", type(rf_scores), getattr(rf_scores, "shape", None))
print("DEBUG: rf_preds type/shape:", type(rf_preds), getattr(rf_preds, "shape", None))
print("DEBUG: iso_scores type/shape:", type(iso_scores), getattr(iso_scores, "shape", None))
print("DEBUG: iso_preds type/shape:", type(iso_preds), getattr(iso_preds, "shape", None))
print("DEBUG: rf_importance type:", type(rf_importance))

# Kontrollera att X_test och y_test finns — annars avbryt
if "X_test" not in globals() or "y_test" not in globals():
    raise RuntimeError("X_test och/eller y_test saknas i miljön. Kör träningscellerna först.")

# ---------- KPI: samla huvudmetrics för RF och ISO (om tillgängliga) ----------
kpi_rows = []

# RF-metrics (om vi har scores eller preds)
if is_nonempty_array_like(rf_scores) or is_nonempty_array_like(rf_preds):
    # beräkna klassiska metrics om preds finns
    acc = prec = rec = f1 = None
    if is_nonempty_array_like(rf_preds):
        preds_arr = np.asarray(rf_preds)
        acc = float(accuracy_score(y_test, preds_arr))
        prec = float(precision_score(y_test, preds_arr, zero_division=0))
        rec = float(recall_score(y_test, preds_arr, zero_division=0))
        f1 = float(f1_score(y_test, preds_arr, zero_division=0))
    # ROC/PR om scores finns
    roc_auc_val = pr_auc_val = None
    if is_nonempty_array_like(rf_scores):
        scores_arr = np.asarray(rf_scores)
        roc_auc_val, _, _ = safe_auc(y_test, scores_arr)
        pr_auc_val, _, _ = safe_pr(y_test, scores_arr)
    kpi_rows.append({
        "Model": "RandomForest",
        "Accuracy": acc,
        "Precision_malicious": prec,
        "Recall_malicious": rec,
        "F1_malicious": f1,
        "ROC_AUC": roc_auc_val,
        "PR_AUC": pr_auc_val
    })
else:
    print("RF: inga giltiga scores eller prediktioner hittades. Kontrollera att rf, y_proba och y_pred finns i miljön.")

# ISO-metrics (vi beräknar ROC/PR och klassiska metrics om preds finns)
if is_nonempty_array_like(iso_scores) or is_nonempty_array_like(iso_preds):
    acc_i = prec_i = rec_i = f1_i = None
    if is_nonempty_array_like(iso_preds):
        preds_iso_arr = np.asarray(iso_preds)
        acc_i = float(accuracy_score(y_test, preds_iso_arr))
        prec_i = float(precision_score(y_test, preds_iso_arr, zero_division=0))
        rec_i = float(recall_score(y_test, preds_iso_arr, zero_division=0))
        f1_i = float(f1_score(y_test, preds_iso_arr, zero_division=0))
    roc_auc_i = pr_auc_i = None
    if is_nonempty_array_like(iso_scores):
        scores_iso_arr = np.asarray(iso_scores)
        roc_auc_i, _, _ = safe_auc(y_test, scores_iso_arr)
        pr_auc_i, _, _ = safe_pr(y_test, scores_iso_arr)
    kpi_rows.append({
        "Model": "IsolationForest",
        "Accuracy": acc_i,
        "Precision_malicious": prec_i,
        "Recall_malicious": rec_i,
        "F1_malicious": f1_i,
        "ROC_AUC": roc_auc_i,
        "PR_AUC": pr_auc_i
    })
else:
    print("ISO: inga giltiga scores eller prediktioner hittades. ISO-visualiseringar kommer att visa tillgängliga plots utan confusion matrix för ISO.")

kpi_df = pd.DataFrame(kpi_rows)
print("✅ KPI-tabell:")
display(kpi_df)

# ---------- ROC & PR subplot (jämför modeller) ----------
fig = make_subplots(rows=1, cols=2, subplot_titles=("ROC Curve","Precision-Recall Curve"))

# RF kurvor
if is_nonempty_array_like(rf_scores):
    rf_scores_arr = np.asarray(rf_scores)
    auc_rf, fpr_rf, tpr_rf = safe_auc(y_test, rf_scores_arr)
    pr_auc_rf, prec_rf, rec_rf = safe_pr(y_test, rf_scores_arr)
    if fpr_rf is not None:
        fig.add_trace(go.Scatter(x=fpr_rf, y=tpr_rf, mode='lines', name=f"RF ROC AUC={auc_rf:.3f}"), row=1, col=1)
    if rec_rf is not None:
        fig.add_trace(go.Scatter(x=rec_rf, y=prec_rf, mode='lines', name=f"RF PR AUC={pr_auc_rf:.3f}"), row=1, col=2)

# ISO kurvor
if is_nonempty_array_like(iso_scores):
    iso_scores_arr = np.asarray(iso_scores)
    auc_iso, fpr_iso, tpr_iso = safe_auc(y_test, iso_scores_arr)
    pr_auc_iso, prec_iso, rec_iso = safe_pr(y_test, iso_scores_arr)
    if fpr_iso is not None:
        fig.add_trace(go.Scatter(x=fpr_iso, y=tpr_iso, mode='lines', name=f"ISO ROC AUC={auc_iso:.3f}"), row=1, col=1)
    if rec_iso is not None:
        fig.add_trace(go.Scatter(x=rec_iso, y=prec_iso, mode='lines', name=f"ISO PR AUC={pr_auc_iso:.3f}"), row=1, col=2)

# Referenslinje för ROC
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', line=dict(dash='dash', color='gray'), showlegend=False), row=1, col=1)
fig.update_xaxes(title_text="False Positive Rate", row=1, col=1)
fig.update_yaxes(title_text="True Positive Rate", row=1, col=1)
fig.update_xaxes(title_text="Recall", row=1, col=2)
fig.update_yaxes(title_text="Precision", row=1, col=2)
fig.update_layout(height=480, width=1100, title_text="ROC & PR – RandomForest vs IsolationForest")
fig.show()

# ---------- Confusion matrix (ENDAST Random Forest) ----------
# Vi tar bort confusion matrix för Isolation Forest enligt önskemål.
preds_rf_thresh = None
if is_nonempty_array_like(rf_scores):
    preds_rf_thresh = (np.asarray(rf_scores) >= threshold).astype(int)
elif is_nonempty_array_like(rf_preds):
    preds_rf_thresh = np.asarray(rf_preds)

if preds_rf_thresh is not None:
    cm = confusion_matrix(y_test, preds_rf_thresh)
    cm_df = pd.DataFrame(cm, index=["Actual 0","Actual 1"], columns=["Pred 0","Pred 1"])
    fig_cm = px.imshow(cm_df, text_auto=True, color_continuous_scale="Blues", title=f"Confusion Matrix (Random Forest) @ threshold {threshold}")
    fig_cm.update_layout(width=600, height=400)
    fig_cm.show()
else:
    print("Ingen binär prediktion tillgänglig för Random Forest confusion matrix. Ange y_pred eller y_proba.")

# ---------- Score-distribution (overlay) för båda modeller ----------
# Visar hur scores fördelar sig per klass; hjälper att bedöma separation.
if is_nonempty_array_like(rf_scores):
    df_rf = pd.DataFrame({"score": np.asarray(rf_scores), "label": y_test.values})
    fig_rf_hist = px.histogram(df_rf, x="score", color="label", nbins=40, barmode="overlay",
                               opacity=0.6, title="RF score distribution (overlay)", labels={"label":"Actual label"})
    fig_rf_hist.update_layout(width=800, height=350)
    fig_rf_hist.show()

if is_nonempty_array_like(iso_scores):
    df_iso = pd.DataFrame({"score": np.asarray(iso_scores), "label": y_test.values})
    fig_iso_hist = px.histogram(df_iso, x="score", color="label", nbins=40, barmode="overlay",
                                opacity=0.6, title="ISO score distribution (overlay)", labels={"label":"Actual label"})
    fig_iso_hist.update_layout(width=800, height=350)
    fig_iso_hist.show()

# ---------- Feature importance (RF) och permutation importance ----------
# RF inbyggd importance (endast RF har denna)
if rf_importance is not None:
    try:
        fi = pd.DataFrame({"feature": rf_importance.index, "importance": rf_importance.values}).head(top_n)
        fig_fi = px.bar(fi[::-1], x="importance", y="feature", orientation="h", title=f"Top {top_n} Feature Importance (RF)")
        fig_fi.update_layout(width=700, height=350)
        fig_fi.show()
    except Exception as e:
        print("Kunde inte rita RF feature importance:", type(e).__name__, e)
else:
    print("RF feature importance saknas eller ej tillgänglig som 'importance' (pd.Series).")

# Permutation importance (valfritt, tungt)
if show_perm and 'rf' in globals() and is_nonempty_array_like(X_test):
    try:
        print("Beräknar permutation importance (kan ta tid)...")
        perm_res = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
        perm_df = pd.DataFrame({
            "feature": X_test.columns,
            "perm_mean": perm_res.importances_mean,
            "perm_std": perm_res.importances_std
        }).sort_values("perm_mean", ascending=False).head(top_n)
        display(perm_df)
        fig_perm = px.bar(perm_df[::-1], x="perm_mean", y="feature", orientation="h", title="Permutation importance (top)")
        fig_perm.update_layout(width=700, height=350)
        fig_perm.show()
    except Exception as e:
        print("Permutation importance misslyckades eller tog för lång tid:", type(e).__name__, e)
elif show_perm:
    print("Permutation importance kräver att RF är tränad och X_test finns i miljön.")

# ---------- Cumulative gains / lift (approx) för RF och ISO ----------
def cumulative_gains(y_true, scores):
    df = pd.DataFrame({"y": y_true, "score": scores})
    df = df.sort_values("score", ascending=False).reset_index(drop=True)
    df["cum_positive"] = df["y"].cumsum()
    total_pos = df["y"].sum() if df["y"].sum() > 0 else 1
    df["cum_pct_pos"] = df["cum_positive"] / total_pos
    df["pct_samples"] = (np.arange(len(df)) + 1) / len(df)
    return df

if is_nonempty_array_like(rf_scores):
    cg_rf = cumulative_gains(y_test, np.asarray(rf_scores))
    fig_cg = go.Figure()
    fig_cg.add_trace(go.Scatter(x=cg_rf["pct_samples"], y=cg_rf["cum_pct_pos"], name="RF"))
    fig_cg.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', line=dict(dash='dash', color='gray'), showlegend=False))
    fig_cg.update_layout(title="Cumulative gains (RF)", xaxis_title="Proportion of sample", yaxis_title="Proportion of positives captured", width=700, height=400)
    fig_cg.show()

if is_nonempty_array_like(iso_scores):
    cg_iso = cumulative_gains(y_test, np.asarray(iso_scores))
    fig_cg2 = go.Figure()
    fig_cg2.add_trace(go.Scatter(x=cg_iso["pct_samples"], y=cg_iso["cum_pct_pos"], name="ISO"))
    fig_cg2.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', line=dict(dash='dash', color='gray'), showlegend=False))
    fig_cg2.update_layout(title="Cumulative gains (ISO)", xaxis_title="Proportion of sample", yaxis_title="Proportion of positives captured", width=700, height=400)
    fig_cg2.show()

# ---------- Avslutande meddelande ----------
print("Dashboard skapad")



Inställningar: model=Both, threshold=0.5, TopN=10, Perm=False
DEBUG: rf_scores type/shape: <class 'numpy.ndarray'> (803,)
DEBUG: rf_preds type/shape: <class 'numpy.ndarray'> (803,)
DEBUG: iso_scores type/shape: <class 'NoneType'> None
DEBUG: iso_preds type/shape: <class 'numpy.ndarray'> (803,)
DEBUG: rf_importance type: <class 'pandas.core.series.Series'>
✅ KPI-tabell:


,Model,Accuracy,Precision_malicious,Recall_malicious,F1_malicious,ROC_AUC,PR_AUC
0,RandomForest,0.996264,1.000000,0.25,0.400000,1.0,1.0
1,IsolationForest,0.960149,0.033333,0.25,0.058824,NaN,NaN


🎉 Dashboard klar. Confusion matrix visas endast för Random Forest enligt önskemål.
- Ändra widgets i toppen och klicka Refresh i Dashboard för att uppdatera vyerna.
- Dela upp cellen i flera celler (KPI, ROC/PR, Confusion (RF), Feature importance, Permutation) om du vill lägga varje figur som separat visualisering i Databricks Dashboard.
